# **Relationship Between Swimming and Running Performance in Triathletes**

**Authors:** Elena Liaño Rengel and Varvara Gkiouzepa Petropoulou

**Public project website:** GitHub Pages — https://elenaliano.github.io/

**Course:** CMPS3160 - Intro to Data Science

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/Data Science Project")

print("Folder exists:", folder.exists())

if folder.exists():
    for file in folder.iterdir():
        print(file.name)

Folder exists: True
Datasets for SWIM + RUN correlation.gdoc
Datasets
Milestone 1 Draft.gdoc
Milestone 1.ipynb


### **Datasets**

Dataset 1: [Ironman 70.3 (2004–2020)](https://www.kaggle.com/datasets/aiaiaidavid/ironman-703-race-data-between-2004-and-2020)

- To what extent is swimming performance associated with running performance among IRONMAN 70.3 triathletes?

- Does the relationship between swimming and running performance vary across age groups and between male and female triathletes?

Dataset 2: [2019 Ironman World Championship](https://www.kaggle.com/datasets/andyesi/2019-ironman-world-championship-results)

- Is the relationship between swimming and running performance different among professional athletes and age-group competitors at the 2019 IRONMAN World Championship?

Dataset 3: [Multi-Sport Race Result Data](https://github.com/matsubo/triathlon-result-data)

- Does the strength of the association between swimming and running performance differ across triathlon race distances?

- Do athletes tend to exhibit more balanced swimming and running performances in shorter or longer triathlon events?

### **Collaboration Plan**

The team plan is to meet once a week or as needed to evaluate progress, work on research questions, and resolve problems. Google Drive is used to have a shared space for the team's documents, and Google Collab to write and run Python code. The most common form of communication will be WhatsApp.

We will be dividing the work based on each Milestone's requirements while considering the stronger aspects of each team member. We both worked on the data collection. Our preference is to work in person when deciding on roles and make sure we are communicating ideas freely while discussing our options. We will be us our shared Drive folder to include any drafts and ideas.

### **Load Dataset 1**

We aqcuired this dataset from kaggle and used its import reccomendation for downloading via kagglehub.

In [ ]:
!pip install kagglehub

import kagglehub

# We download the latest version from the webpage of our dataset
path = kagglehub.dataset_download(
    "aiaiaidavid/ironman-703-race-data-between-2004-and-2020"
)

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ironman-703-race-data-between-2004-and-2020' dataset.
Path to dataset files: /kaggle/input/ironman-703-race-data-between-2004-and-2020


In [ ]:
import pandas as pd

df = pd.read_csv("/root/.cache/kagglehub/datasets/aiaiaidavid/ironman-703-race-data-between-2004-and-2020/versions/5/Half_Ironman_df6.csv")

display(df.head())
df.info()

,Gender,AgeGroup,AgeBand,Country,CountryISO2,EventYear,EventLocation,SwimTime,Transition1Time,BikeTime,Transition2Time,RunTime,FinishTime
0,M,40-44,40,Andorra,AD,2019,IRONMAN 70.3 South American Championship Bueno...,1679,119,9107,95,5515,16514
1,M,45-49,45,Andorra,AD,2019,IRONMAN 70.3 South American Championship Bueno...,2070,177,9160,132,6070,17609
2,M,45-49,45,Andorra,AD,2020,IRONMAN 70.3 Bariloche,1667,161,9891,122,5190,17031
3,M,45-49,45,Andorra,AD,2019,IRONMAN 70.3 World Championship,1750,183,10363,160,5071,17527
4,M,40-44,40,Andorra,AD,2019,IRONMAN 70.3 World Championship,2063,182,10065,142,5556,18008


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 840075 entries, 0 to 840074
Data columns (total 13 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   Gender           840075 non-null  object
 1   AgeGroup         840075 non-null  object
 2   AgeBand          840075 non-null  int64 
 3   Country          840075 non-null  object
 4   CountryISO2      840075 non-null  object
 5   EventYear        840075 non-null  int64 
 6   EventLocation    840075 non-null  object
 7   SwimTime         840075 non-null  int64 
 8   Transition1Time  840075 non-null  int64 
 9   BikeTime         840075 non-null  int64 
 10  Transition2Time  840075 non-null  int64 
 11  RunTime          840075 non-null  int64 
 12  FinishTime       840075 non-null  int64 
dtypes: int64(8), object(5)
memory usage: 83.3+ MB


### **ETL: transform, validate, and create a tidy table**


We will do some transformation to our table according to tidy data.

We want to rename our columns in a way that it will be easier for reading and understanding. For the variables measured in seconds, SwimTime, Transition1Time, BikeTime, Transition2Time, RunTime, FinishTime, we will exclude the keyword time for space purposes.

In [ ]:
column_names = {
    "Gender": "Gender",
    "AgeGroup": "Age_Group",
    "AgeBand": "Age_Band",
    "Country": "Country",
    "CountryISO2": "Country_Code",
    "EventYear": "Year",
    "EventLocation": "Event",
    "SwimTime": "Swim",
    "Transition1Time": "Transition_1",
    "BikeTime": "Bike",
    "Transition2Time": "Transition_2",
    "RunTime": "Run",
    "FinishTime": "Finish"
}

df = df.rename(columns=column_names)

In order for our data sets to be more readable, we also notice that "IRONMAN 70.3" is part of 'Event' for each row, so we strip it. Since this dataset only includes Ironman 70.3, we are more interested in seeing the place of the event or if it is a world champioship or not.

Moreover, we sort the values according to event and year so that it is easier to manipulate.

In [ ]:
df["Event"] = (
    df["Event"]
    .str.replace(r"(?i)^IRONMAN 70\.3\s*", "", regex=True)
    .str.strip(" -,:")
)

df = df.sort_values(
    ["Event", "Year"]
).reset_index(drop=True)

display(df.head())

,Gender,Age_Group,Age_Band,Country,Country_Code,Year,Event,Swim,Transition_1,Bike,Transition_2,Run,Finish
0,M,30-34,30,Austria,AT,2014,Aarhus,2034,244,9448,180,5876,17782
1,M,25-29,25,Austria,AT,2014,Aarhus,2275,259,10476,145,5956,19111
2,M,50-54,50,Austria,AT,2014,Aarhus,2542,403,10002,221,6382,19550
3,M,50-54,50,Austria,AT,2014,Aarhus,2828,375,11704,232,8112,23251
4,M,25-29,25,Australia,AU,2014,Aarhus,1874,166,9256,138,4833,16267


Now we check for 0 values. While usually, we prefer to not have NaN values, here it is important to get rid of the 0 values, since they will mess up our calculations. Thus, we will turn all 0 values into NaN. We notice that 'Age Band' has zero values, so we replace them with NaN.

In [ ]:
print("\nzeros:")
print((df.select_dtypes(include="number") == 0).sum())

import numpy as np

df = df.replace(0, np.nan)

print("\nzeros:")
print((df.select_dtypes(include="number") == 0).sum())



zeros:
Age_Band        16611
Year                0
Swim                0
Transition_1        0
Bike                0
Transition_2        0
Run                 0
Finish              0
dtype: int64

zeros:
Age_Band        0
Year            0
Swim            0
Transition_1    0
Bike            0
Transition_2    0
Run             0
Finish          0
dtype: int64


In this part we want to work on Age Group. Since Age Group is an object type, it is not helpful for calculations. Moreover, it has some values of 00. We want to deal with both those problems.

First, we decided to create a new column using the Age Group column. We will name it Age_Group_Mean, and it will get the mean value of each age group. For example, if the age group is 30-34, our mean will be 32. Thus, this column will also be an integer and easier to work on. We want to get rid of the '0' values also, so we will turn them into NaN values as we did above.

We understand and keep in mind that this column doesn't tell us the exact age of the triathletes, but categorizes all athletes from each age group in a way that is easier for us to handle.

In [ ]:
import numpy as np
import pandas as pd

df["Age_Group_Mean"] = (
    df["Age_Group"]
    .str.split("-", expand=True)[[0, 1]]
    .astype(float)
    .mean(axis=1)
    .round()
    .astype("float")
)

df["Age_Group_Mean"] = df["Age_Group_Mean"].replace(0, np.nan)

display(df[["Age_Group", "Age_Group_Mean"]].head(10))

,Age_Group,Age_Group_Mean
0,30-34,32.0
1,25-29,27.0
2,50-54,52.0
3,50-54,52.0
4,25-29,27.0
5,45-49,47.0
6,35-39,37.0
7,45-49,47.0
8,45-49,47.0
9,35-39,37.0


In [ ]:
df.insert(
    df.columns.get_loc("Age_Band"),
    "Age_Group_Mean",
    df.pop("Age_Group_Mean")
)

df = df.drop(columns=["Age_Group"])

Here we check for duplicate entries. We notice there is 17 duplicate rows. We check that they are duplicates all across the columns and then delete one of the two same.

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

df[df.duplicated(keep=False)].sort_values(
    ["Event", "Year", "Finish"]
)

df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 17
Duplicate rows: 0


In [ ]:
df.head()


,Gender,Age_Group_Mean,Age_Band,Country,Country_Code,Year,Event,Swim,Transition_1,Bike,Transition_2,Run,Finish
0,M,32.0,30.0,Austria,AT,2014,Aarhus,2034,244,9448,180,5876,17782
1,M,27.0,25.0,Austria,AT,2014,Aarhus,2275,259,10476,145,5956,19111
2,M,52.0,50.0,Austria,AT,2014,Aarhus,2542,403,10002,221,6382,19550
3,M,52.0,50.0,Austria,AT,2014,Aarhus,2828,375,11704,232,8112,23251
4,M,27.0,25.0,Australia,AU,2014,Aarhus,1874,166,9256,138,4833,16267


We will split our data set into two. One set for women and one set for men. Since gender plays a big role in performance, we can compare the relationship in running and swimming for men and women seperately. We also reset the index, so that both tables have indexes 0,1... and not just the indexes from the original table.

In [ ]:
print(df["Gender"].unique())
men_df = df[df["Gender"] == "M"].copy()
women_df = df[df["Gender"] == "F"].copy()

print("Men's dataset:", men_df.shape) # we split our data into two different tables one for women one for men
print("Women's dataset:", women_df.shape)

men_df = men_df.reset_index(drop=True)
women_df = women_df.reset_index(drop=True)

men_df["Gender"] = "Male"
women_df["Gender"] = "Female"

display(men_df.head())
display(women_df.head())

['M' 'F']
Men's dataset: (635667, 13)
Women's dataset: (204391, 13)


,Gender,Age_Group_Mean,Age_Band,Country,Country_Code,Year,Event,Swim,Transition_1,Bike,Transition_2,Run,Finish
0,Male,32.0,30.0,Austria,AT,2014,Aarhus,2034,244,9448,180,5876,17782
1,Male,27.0,25.0,Austria,AT,2014,Aarhus,2275,259,10476,145,5956,19111
2,Male,52.0,50.0,Austria,AT,2014,Aarhus,2542,403,10002,221,6382,19550
3,Male,52.0,50.0,Austria,AT,2014,Aarhus,2828,375,11704,232,8112,23251
4,Male,27.0,25.0,Australia,AU,2014,Aarhus,1874,166,9256,138,4833,16267


,Gender,Age_Group_Mean,Age_Band,Country,Country_Code,Year,Event,Swim,Transition_1,Bike,Transition_2,Run,Finish
0,Female,47.0,45.0,Canada,CA,2014,Aarhus,1988,292,10904,154,5042,18380
1,Female,47.0,45.0,Canada,CA,2014,Aarhus,2154,419,13684,319,8080,24656
2,Female,37.0,35.0,Switzerland,CH,2014,Aarhus,2506,390,10763,293,7046,20998
3,Female,NaN,NaN,Germany,DE,2014,Aarhus,1973,205,9361,131,5306,16976
4,Female,NaN,NaN,Germany,DE,2014,Aarhus,1787,170,9989,116,5068,17130


### **Challenges**


*   There was 0 values in age that would mess up our calculations if we didn't change them to NaN
*   Duplicate rows had to be removed
*   Do not know exact age of triathletes
*   Age Group was given as an object which was not helpful for our calculations
*   In our first approach, we split the dataset into men and women first and then applied all the changes which took much more space than intended



### **Summary Statistics**


*   There is more than 3 times more entries for me than women.
*   List item

